# chip-lakehouse: exploration

Interactive scratch space for poking at the lakehouse - closer to a Databricks-notebook workflow than re-running a whole script to check one thing.

**This notebook is for exploration only.** No pipeline logic lives here - it imports and calls the same `src/` modules the real pipeline stages use (`bronze_ingest.py`, `silver_transform.py`, `gold_marts.py`). See `docs/standard.md` for why the two stay separate.

Requires the Docker Unity Catalog server to be running: `cd docker && docker compose up -d`.

In [1]:
import sys
from pathlib import Path

# notebooks/ is a sibling of src/, not inside it - add src/ to the path so
# the pipeline modules import the same way they do when run as scripts.
sys.path.insert(0, str(Path.cwd().parent / "src"))

from spark_session import get_spark, CATALOG_NAME

spark = get_spark()
spark

## What's in the catalog

In [2]:
for schema in ["bronze", "silver", "gold", "ml"]:
    print(f"=== {schema} ===")
    spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{schema}").show(truncate=False)

=== bronze ===


+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|bronze   |accounts     |false      |
|bronze   |savings_goals|false      |
|bronze   |transactions |false      |
|bronze   |users        |false      |
+---------+-------------+-----------+

=== silver ===
+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|silver   |accounts     |false      |
|silver   |savings_goals|false      |
|silver   |transactions |false      |
|silver   |users        |false      |
+---------+-------------+-----------+

=== gold ===
+---------+---------------+-----------+
|namespace|tableName      |isTemporary|
+---------+---------------+-----------+
|gold     |account_summary|false      |
|gold     |customer_360   |false      |
+---------+---------------+-----------+

=== ml ===
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+-------

## Gold marts

`customer_360` is built by reading `account_summary` back from the catalog - see `src/gold_marts.py`'s module docstring for why that makes the two marts provably consistent with each other.

In [3]:
spark.table(f"{CATALOG_NAME}.gold.customer_360").orderBy("user_id").limit(10).toPandas()

26/09/05 00:31:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,user_id,age_band,signup_date,num_accounts,total_balance,total_inflows,total_outflows,num_savings_goals,total_target_amount,total_saved_toward_goals,goal_progress_pct
0,00323a26-af72-4349-ba84-c5fe3492d6de,20-29,2025-06-28,3,3159.43,9828.41,6668.98,0,0.00,0.00,None
1,0056fa27-0168-46f9-bdc5-4e89fdc38975,50-59,2025-12-02,2,10552.46,17732.17,7179.71,1,17948.16,14181.04,79.0
2,006e8ae8-a8f3-4763-aad2-4b4d789d7610,70-79,2025-12-17,3,18176.34,30531.11,12354.77,1,9358.20,2468.47,26.4
3,007af90b-6529-4a26-a0a8-1808a757d5ee,70-79,2025-04-09,3,23587.73,40978.50,17390.77,1,15954.65,8862.18,55.5
4,009a985b-0072-4321-abc4-a10abbdc7c16,30-39,2025-01-20,3,8616.99,12223.28,3606.29,0,0.00,0.00,None
5,01278327-767a-4699-a3f5-b0151b567fae,20-29,2025-03-21,2,10078.83,20911.59,10832.76,0,0.00,0.00,None
6,012a6bf4-e28e-4ecf-97f3-90920c876014,40-49,2026-02-20,1,3954.23,4443.49,489.26,1,1031.87,890.11,86.3
7,0145c00f-62ad-4b6e-9f36-5368f5c81ba3,30-39,2024-12-15,2,5993.89,6936.12,942.23,1,17956.00,8767.46,48.8
8,0147da3f-8913-48fa-b530-5858e8f50755,60-69,2026-03-15,2,11933.14,19987.19,8054.05,0,0.00,0.00,None
9,01786986-68fc-453a-8778-5494b3688db4,20-29,2024-07-22,1,11408.56,16604.85,5196.29,1,9995.51,7479.39,74.8


In [4]:
spark.table(f"{CATALOG_NAME}.gold.account_summary").orderBy("account_id").limit(10).toPandas()

,account_id,user_id,account_type,opened_date,total_inflows,total_outflows,balance,transaction_count,first_transaction_ts,last_transaction_ts
0,001426ad-ce8b-49c9-a51e-30e26ba25b48,f58cea3d-a475-40eb-a114-c041ba8bf0bd,pension,2026-02-28,9119.81,8739.84,379.97,48,2026-03-06 03:33:27.723151,2026-08-30 19:38:17.400791
1,001ef9f5-0b25-407e-8c1a-c5f521fc82d0,b4a6d03c-f2b7-4b0e-acbb-8545d4004928,pension,2025-11-12,1156.14,1600.12,-443.98,6,2026-01-01 04:09:44.226895,2026-08-28 14:17:25.198081
2,00241917-4409-4c3e-a522-dbe9cfa10367,3f13ff6d-46cb-4cf6-b543-97320002cea1,pension,2024-04-08,1177.71,855.12,322.59,6,2024-09-30 18:13:21.079171,2026-08-06 11:52:45.684080
3,0036c4ac-163b-4084-a06e-bca654d11471,3ed8714a-5105-4a28-8fab-96be98fb332b,investment,2025-11-08,6309.48,5870.91,438.57,24,2025-11-08 05:30:48.242562,2026-08-07 08:04:43.559868
4,0039c645-c824-481f-bae8-019ff0085216,1b71fda5-dcd8-4738-8add-66faa104d346,savings,2025-11-19,10968.66,5386.06,5582.60,35,2025-11-29 21:05:46.111436,2026-08-30 21:06:34.253110
5,005b53b2-4c6c-43ad-a6df-c6292f244edd,86e30aed-2394-4f0e-945f-57dd130096af,pension,2026-08-25,6610.01,1828.53,4781.48,20,2026-08-25 10:11:09.633526,2026-08-31 12:09:50.595813
6,005d4fdb-de9e-4c80-9b6b-344599e78afb,d8d2a57c-5289-4e95-965d-4bd3cae58af2,investment,2025-06-11,19272.97,6029.44,13243.53,42,2025-06-16 13:14:50.835535,2026-08-18 12:49:27.532431
7,0060c723-416b-4ebd-b5bc-fe153a954091,8ea24f29-2614-4a81-a43c-3227c8d06f70,pension,2024-03-13,11489.20,407.13,11082.07,15,2024-05-09 14:16:55.587821,2026-03-13 15:03:05.252046
8,007010f8-4a1d-4d19-bee7-d221704bf519,eda3ac97-83be-4a0e-b1e8-0cfa7f1ccead,pension,2025-04-25,5147.17,1838.46,3308.71,14,2025-05-01 15:13:58.138536,2026-07-12 14:05:00.991907
9,008df0c0-a27e-43cd-a954-862622869abf,e98e3f79-6204-4e41-a1c7-ebbe7032207c,savings,2024-12-23,2126.03,1711.37,414.66,10,2024-12-23 21:04:37.784814,2026-07-13 12:36:28.091828


## Scratch

Anything ad hoc goes below - a quick `spark.sql(...)`, a `.toPandas()` for a plot, checking a hunch before it becomes real code in `src/`.